# TP 2 — « L'IA lit 40 ans d'images de ma ville »
### Entraîner un modèle de Machine Learning à reconnaître le bâti, puis l'appliquer à 40 ans d'archives Landsat
*Sans compte ni inscription : tout se passe dans Google Colab, à partir de catalogues d'images en libre accès.*

**Durée :** 1h30 — **Niveau :** aucun prérequis en programmation

**Ce que vous allez réaliser**
1. Une animation de votre ville de 1984 à aujourd'hui (l'accroche : ce sont les données brutes).
2. **Un modèle d'IA que vous entraînez vous-même** pour distinguer le bâti du non-bâti.
3. L'évaluation de ce modèle : précision, matrice de confusion, variables les plus utiles.
4. L'application du modèle aux 40 ans d'images : **courbe de la surface bâtie**, **carte de l'âge des quartiers** et animation de l'étalement urbain.

## Où est l'IA dans ce TP ?

| Étape | Ce qu'on fait | Est-ce de l'IA ? |
|---|---|---|
| 2 | Rassembler une image sans nuage par an, faire l'animation | Non : traitement d'image classique |
| 3 | Préparer les **étiquettes** (la « vérité » que le modèle doit apprendre) | Préparation des données d'apprentissage |
| 4 | Calculer les **variables** décrivant chaque pixel | Préparation des données d'apprentissage |
| 5 | Séparer les pixels d'**entraînement** et de **test** | Oui : protocole d'apprentissage |
| 6 | **Entraîner** le modèle (Random Forest) | Oui : apprentissage supervisé |
| 7 | **Évaluer** le modèle sur des pixels jamais vus | Oui : validation |
| 8 | **Appliquer** le modèle à chaque année depuis 1984 | Oui : prédiction (inférence) |
| 9 | Corriger les incohérences dans le temps, produire courbe et cartes | Post-traitement et aide à la décision |

Vous retrouverez à chaque étape un encadré **« Étape IA »** qui explique ce que fait la machine.

---
Pour exécuter une cellule : cliquer dessus puis **Maj + Entrée**. Exécutez les cellules dans l'ordre.

## Étape 0 — Installation et connexion au catalogue (≈ 1 minute)

In [ ]:
%pip install -q pystac-client planetary-computer rasterio shapely leafmap folium

In [ ]:
import os, math, time, warnings
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from PIL import Image as PILImage, ImageDraw, ImageFont
from IPython.display import Image, display

import rasterio
from rasterio.crs import CRS
from rasterio.enums import Resampling
from rasterio.transform import from_origin, array_bounds
from rasterio.vrt import WarpedVRT
from rasterio.warp import transform_bounds, calculate_default_transform, reproject
from shapely.geometry import box, shape

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

import folium
import leafmap
import pystac_client
import planetary_computer

os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "EMPTY_DIR"
os.environ["GDAL_HTTP_MAX_RETRY"] = "3"
os.environ["GDAL_HTTP_RETRY_DELAY"] = "1"
warnings.filterwarnings("ignore")

catalogue = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)
print("Connexion au catalogue réussie.")

## Étape 1 — Choisir la zone d'étude
La carte s'ouvre sur **Tanger**, une des villes marocaines qui a le plus changé en 40 ans.

- Pour garder Tanger : ne dessinez rien et passez à la cellule suivante.
- Pour étudier **votre** ville : déplacez la carte puis dessinez un rectangle avec l'outil ▭ (barre de gauche).

**Conseil :** un rectangle de 10 à 25 km de côté, qui contient à la fois de la ville et de la campagne.

In [ ]:
m = leafmap.Map(center=[35.755, -5.825], zoom=11, height="550px")
m.add_basemap("SATELLITE")
m

In [ ]:
NOM_ZONE = "Tanger"                              # utilisé dans les titres
BBOX_DEFAUT = [-5.93, 35.70, -5.72, 35.81]       # [ouest, sud, est, nord]

bbox = m.user_roi_bounds()
if bbox is None:
    bbox = BBOX_DEFAUT
    print("Zone par défaut :", NOM_ZONE)
else:
    print("Zone dessinée sur la carte. Pensez à modifier NOM_ZONE ci-dessus.")

largeur_km = (bbox[2] - bbox[0]) * 111 * math.cos(math.radians((bbox[1] + bbox[3]) / 2))
hauteur_km = (bbox[3] - bbox[1]) * 111
print(f"Dimensions : {largeur_km:.1f} km x {hauteur_km:.1f} km, soit {largeur_km * hauteur_km * 100:,.0f} ha")
if max(largeur_km, hauteur_km) > 30:
    print("ATTENTION : zone trop grande, redessinez un rectangle plus petit.")

## Étape 2 — Les données : une image Landsat sans nuage par an depuis 1984
**Pas encore d'IA ici.** Pour chaque année, le programme cherche les images de mai à septembre les moins nuageuses, ne lit que le morceau qui couvre votre zone, masque les nuages et calcule l'image médiane.

On conserve **6 bandes spectrales** : bleu, vert, rouge (visibles à l'œil) et proche infrarouge, moyen infrarouge 1 et 2 (invisibles à l'œil, mais très utiles au modèle).

Les deux cellules suivantes définissent les outils. Exécutez-les sans les modifier.

In [ ]:
# Outils 1/2 : grille de travail, lecture à distance, choix des scènes
def creer_grille(bbox, resolution):
    lon, lat = (bbox[0] + bbox[2]) / 2, (bbox[1] + bbox[3]) / 2
    crs = CRS.from_epsg((32600 if lat >= 0 else 32700) + int((lon + 180) // 6) + 1)
    o, s, e, n = transform_bounds("EPSG:4326", crs, *bbox)
    pas = 30   # les grilles à 10 m et 30 m partagent ainsi la même emprise
    o, s = math.floor(o / pas) * pas, math.floor(s / pas) * pas
    e, n = math.ceil(e / pas) * pas, math.ceil(n / pas) * pas
    return {"crs": crs, "res": resolution, "bounds": (o, s, e, n),
            "transform": from_origin(o, n, resolution, resolution),
            "width": int(round((e - o) / resolution)),
            "height": int(round((n - s) / resolution))}

def lire(adresse, grille, methode=Resampling.bilinear, nodata=0, essais=3):
    for essai in range(essais):
        try:
            with rasterio.open(adresse) as src:
                with WarpedVRT(src, crs=grille["crs"], transform=grille["transform"],
                               width=grille["width"], height=grille["height"],
                               resampling=methode, src_nodata=nodata, nodata=nodata) as vrt:
                    return vrt.read()
        except Exception:
            if essai == essais - 1:
                raise
            time.sleep(2)

ZONE = box(*bbox)
GRILLE_30 = creer_grille(bbox, 30)   # Landsat
GRILLE_10 = creer_grille(bbox, 10)   # carte de référence
HA_PAR_PIXEL = 30 * 30 / 10000       # un pixel Landsat = 0,09 ha

def chercher(collection, debut, fin, nuages_max):
    recherche = catalogue.search(collections=[collection], bbox=bbox, datetime=f"{debut}/{fin}",
                                 query={"eo:cloud_cover": {"lt": nuages_max}}, limit=250)
    return list(recherche.items())

def couverture(scene):
    try:
        return shape(scene.geometry).buffer(0).intersection(ZONE).area / ZONE.area
    except Exception:
        return 0.0

def capteur_utilisable(scene):
    # Landsat 7 : images rayées après 2003, gardé seulement quand il n'y a pas d'alternative
    if scene.properties.get("platform") == "landsat-7":
        return scene.datetime.year in (1999, 2000, 2001, 2002, 2012)
    return True

def choisir_scenes(scenes, annee, nombre, mois=(5, 9)):
    candidates = [s for s in scenes if s.datetime.year == annee and capteur_utilisable(s)]
    en_saison = [s for s in candidates if mois[0] <= s.datetime.month <= mois[1]]
    candidates = en_saison or candidates
    notes = [(couverture(s), s.properties.get("eo:cloud_cover", 100), s) for s in candidates]
    completes = sorted([n for n in notes if n[0] >= 0.99], key=lambda n: n[1])
    partielles = sorted([n for n in notes if 0.2 <= n[0] < 0.99], key=lambda n: (-n[0], n[1]))
    return [n[2] for n in (completes + partielles)[:nombre]]

BANDES = ["blue", "green", "red", "nir08", "swir16", "swir22"]
NOMS_BANDES = ["Bleu", "Vert", "Rouge", "Proche infrarouge", "Moyen infrarouge 1", "Moyen infrarouge 2"]

def lire_scene_landsat(scene):
    try:
        pile = np.stack([lire(scene.assets[b].href, GRILLE_30)[0] for b in BANDES])
        qa = lire(scene.assets["qa_pixel"].href, GRILLE_30, Resampling.nearest)[0]
    except Exception:
        return None
    invalide = (qa == 0) | ((qa & 0b11011) != 0) | (pile == 0).any(axis=0)
    reflectance = pile.astype("float32") * 0.0000275 - 0.2
    reflectance[:, invalide] = np.nan
    return reflectance

def composite_annuel(scenes):
    lues = [r for r in map(lire_scene_landsat, scenes) if r is not None]
    return np.nanmedian(np.stack(lues), axis=0) if lues else None

print(f"Grille Landsat : {GRILLE_30['width']} x {GRILLE_30['height']} pixels de 30 m "
      f"= {GRILLE_30['width'] * GRILLE_30['height']:,} pixels à classer par année")

In [ ]:
# Outils 2/2 : mise en couleurs et animation
def en_couleurs(image, mini=0.02, maxi=0.30, gamma=1.3):
    """Image rouge-vert-bleu affichable à partir des 6 bandes."""
    rvb = image[[2, 1, 0]]
    x = np.clip((rvb - mini) / (maxi - mini), 0, 1) ** (1 / gamma)
    return np.nan_to_num(np.moveaxis(x, 0, -1) * 255).astype("uint8")

def _police(taille):
    try:
        return ImageFont.load_default(size=taille)
    except TypeError:
        return ImageFont.load_default()

def creer_gif(images, etiquettes, fichier, titre="", ips=3, taille=720):
    import unicodedata   # la police intégrée ne connaît pas les accents
    titre = unicodedata.normalize("NFKD", titre).encode("ascii", "ignore").decode()
    vues = []
    for i, (tableau, etiquette) in enumerate(zip(images, etiquettes)):
        im = PILImage.fromarray(tableau)
        echelle = taille / max(im.size)
        im = im.resize((round(im.width * echelle), round(im.height * echelle)), PILImage.LANCZOS)
        dessin = ImageDraw.Draw(im)
        dessin.rectangle([0, 0, im.width, 44], fill=(0, 0, 0))
        dessin.text((10, 8), str(etiquette), fill=(255, 255, 255), font=_police(28))
        decalage = 30 + dessin.textlength(str(etiquette), font=_police(28))
        dessin.text((decalage, 14), titre, fill=(200, 200, 200), font=_police(18))
        dessin.rectangle([0, im.height - 8, int(im.width * (i + 1) / len(images)), im.height],
                         fill=(255, 200, 0))
        vues.append(im)
    vues[0].save(fichier, save_all=True, append_images=vues[1:], duration=int(1000 / ips), loop=0)
    print(f"Animation enregistrée : {fichier} ({len(vues)} images)")
    return fichier

### Charger les 40 ans d'images
- `SCENES_PAR_AN` : nombre d'images combinées chaque année. `1` = plus rapide, `3` = moins de nuages.

Le chargement prend quelques minutes : une ligne s'affiche par année.

In [ ]:
ANNEE_DEBUT = 1984
ANNEE_FIN = 2026
SCENES_PAR_AN = 2

print("Recherche des images Landsat dans le catalogue...")
scenes_landsat = chercher("landsat-c2-l2", f"{ANNEE_DEBUT}-01-01", f"{ANNEE_FIN}-12-31", nuages_max=30)
print(f"{len(scenes_landsat)} scènes peu nuageuses trouvées depuis {ANNEE_DEBUT}.")

toutes_annees = list(range(ANNEE_DEBUT, ANNEE_FIN + 1))
choix = {a: choisir_scenes(scenes_landsat, a, SCENES_PAR_AN) for a in toutes_annees}

def traiter(annee):
    return annee, (composite_annuel(choix[annee]) if choix[annee] else None)

images = {}      # année -> image à 6 bandes
debut = time.time()
with ThreadPoolExecutor(max_workers=8) as executeur:
    for annee, composite in executeur.map(traiter, toutes_annees):
        valide = 0 if composite is None else float(np.isfinite(composite).all(axis=0).mean())
        if valide >= 0.6:
            images[annee] = composite
            capteurs = sorted({s.properties.get("platform", "?") for s in choix[annee]})
            print(f"{annee} : {len(choix[annee])} scène(s), {', '.join(capteurs)}, zone couverte à {valide:.0%}")
        else:
            print(f"{annee} : pas d'image exploitable, année écartée")
annees = sorted(images)
print(f"Terminé en {time.time() - debut:.0f} s : {len(annees)} années sur {len(toutes_annees)}.")

gif_landsat = creer_gif([en_couleurs(images[a]) for a in annees], annees, "timelapse_landsat.gif",
                        titre=f"{NOM_ZONE} - Landsat (30 m)", ips=3)
display(Image(filename=gif_landsat))

**Regardez l'animation.** Vous voyez la ville grandir, mais vous ne pouvez pas répondre à des questions simples : *combien d'hectares bâtis en 1990 ? en 2010 ? quels quartiers datent de quelle période ?*

Tracer le bâti à la main sur 40 images prendrait des semaines. C'est ce travail que nous allons confier à un modèle d'IA.

## Étape 3 — Les étiquettes : ce que le modèle doit apprendre

> **Étape IA — Apprentissage supervisé.** Un modèle supervisé apprend à partir d'**exemples corrigés** : des pixels dont on connaît déjà la bonne réponse (« bâti » ou « non bâti »). Ces réponses s'appellent les **étiquettes**. La qualité du modèle dépend directement de la qualité des étiquettes.

D'où viennent nos étiquettes ? D'une carte d'occupation du sol à 10 m de l'année de référence, elle-même produite par un modèle de **Deep Learning** (Impact Observatory / Esri, à partir de Sentinel-2). Notre modèle va donc apprendre d'un autre modèle : c'est rapide, mais la « vérité » n'est pas parfaite (précision annoncée supérieure à 75 %).

Un pixel Landsat (30 m) contient 9 pixels de cette carte (10 m). On ne garde que les cas **nets** :
- **bâti** : au moins 6 des 9 petits pixels sont bâtis ;
- **non bâti** : au plus 1 des 9 ;
- les pixels **mixtes** sont écartés de l'apprentissage.

In [ ]:
ANNEE_REF = 2023      # année de la carte de référence (disponible de 2017 à 2023)
CODE_BATI = 7

def carte_reference(annee):
    tuiles = list(catalogue.search(collections=["io-lulc-annual-v02"], bbox=bbox,
                                   datetime=f"{annee}-02-01/{annee}-11-30").items())
    tuiles = [t for t in tuiles
              if str(t.properties.get("start_datetime", str(annee)))[:4] == str(annee)]
    if not tuiles:
        raise ValueError(f"Aucune carte de référence pour {annee} (années disponibles : 2017 à 2023).")
    carte = np.zeros((GRILLE_10["height"], GRILLE_10["width"]), dtype="uint8")
    for tuile in tuiles:
        morceau = lire(tuile.assets["data"].href, GRILLE_10, Resampling.nearest)[0]
        carte = np.where(carte == 0, morceau, carte)
    return carte

if ANNEE_REF not in images:
    raise ValueError(f"Pas d'image Landsat pour {ANNEE_REF} : choisissez une autre ANNEE_REF parmi {annees}.")

reference_10m = carte_reference(ANNEE_REF)
H, W = GRILLE_30["height"], GRILLE_30["width"]
blocs = reference_10m.reshape(H, 3, W, 3)
part_bati = (blocs == CODE_BATI).mean(axis=(1, 3))       # part de bâti dans chaque pixel de 30 m
connu = (blocs != 0).all(axis=(1, 3))

etiquettes = np.full((H, W), -1, dtype="int8")           # -1 = mixte ou inconnu
etiquettes[connu & (part_bati >= 6 / 9)] = 1             #  1 = bâti
etiquettes[connu & (part_bati <= 1 / 9)] = 0             #  0 = non bâti

for valeur, nom in [(1, "bâti"), (0, "non bâti"), (-1, "mixtes, écartés")]:
    n = int((etiquettes == valeur).sum())
    print(f"Pixels {nom:16s}: {n:8,} ({n / etiquettes.size:.0%})")

fig, axes = plt.subplots(1, 2, figsize=(16, 8 * H / W + 0.6))
axes[0].imshow(en_couleurs(images[ANNEE_REF]))
axes[0].set_title(f"Image Landsat {ANNEE_REF}")
axes[1].imshow(etiquettes, cmap=ListedColormap(["#9e9e9e", "#2e7d32", "#c4281b"]), vmin=-1, vmax=1,
               interpolation="nearest")
axes[1].set_title(f"Étiquettes {ANNEE_REF} : ce que le modèle doit apprendre")
axes[1].legend(handles=[Patch(facecolor="#c4281b", label="Bâti"), Patch(facecolor="#2e7d32", label="Non bâti"),
                        Patch(facecolor="#9e9e9e", label="Mixte (écarté)")], loc="lower right")
for ax in axes:
    ax.set_axis_off()
plt.tight_layout()
plt.show()

## Étape 4 — Les variables : comment le modèle « voit » un pixel

> **Étape IA — Variables explicatives.** Le modèle ne voit pas une image : pour chaque pixel, il reçoit une **ligne de chiffres**. Ici, 9 chiffres : la réflectance dans les 6 bandes, et 3 **indices** calculés à partir de ces bandes, qui font ressortir la végétation, le bâti et l'eau. Choisir de bonnes variables, c'est apporter au modèle la connaissance du métier.

| Indice | Formule | Ce qu'il fait ressortir |
|---|---|---|
| NDVI | (PIR − Rouge) / (PIR + Rouge) | la végétation |
| NDBI | (MIR1 − PIR) / (MIR1 + PIR) | les surfaces minérales et bâties |
| MNDWI | (Vert − MIR1) / (Vert + MIR1) | l'eau |

In [ ]:
NOMS_VARIABLES = NOMS_BANDES + ["NDVI (végétation)", "NDBI (bâti)", "MNDWI (eau)"]

def variables(image):
    """Transforme une image à 6 bandes en tableau : une ligne par pixel, 9 colonnes."""
    bleu, vert, rouge, pir, mir1, mir2 = image
    ndvi = (pir - rouge) / (pir + rouge + 1e-6)
    ndbi = (mir1 - pir) / (mir1 + pir + 1e-6)
    mndwi = (vert - mir1) / (vert + mir1 + 1e-6)
    return np.stack([bleu, vert, rouge, pir, mir1, mir2, ndvi, ndbi, mndwi], axis=-1).reshape(-1, 9)

X_ref = variables(images[ANNEE_REF])
y_ref = etiquettes.ravel()
exemple = pd.DataFrame(X_ref[y_ref >= 0][:5], columns=NOMS_VARIABLES).round(3)
exemple["Étiquette"] = ["bâti" if v == 1 else "non bâti" for v in y_ref[y_ref >= 0][:5]]
print("Voici ce que le modèle reçoit (5 pixels sur", f"{len(X_ref):,}) :")
exemple

## Étape 5 — Séparer l'entraînement et le test

> **Étape IA — Ne jamais évaluer un modèle sur ce qu'il a déjà vu.** On met de côté une partie des pixels étiquetés : le modèle ne les verra pas pendant l'apprentissage, et ils serviront à le noter. C'est l'équivalent d'un examen avec des questions nouvelles.

Deux précautions :
- **Équilibrer les classes** : on tire autant de pixels bâtis que non bâtis, sinon le modèle apprendrait surtout la classe majoritaire.
- **Séparer par blocs de 1 km** : deux pixels voisins se ressemblent beaucoup. Si l'un est à l'entraînement et l'autre au test, l'examen est trop facile. On réserve donc des **blocs entiers** au test (30 % des blocs).

In [ ]:
PIXELS_PAR_CLASSE = 6000
hasard = np.random.default_rng(42)

utilisable = (y_ref >= 0) & np.isfinite(X_ref).all(axis=1)
lignes, colonnes = np.divmod(np.arange(H * W), W)
bloc = (lignes // 33) * 10000 + (colonnes // 33)            # blocs d'environ 1 km
blocs_uniques = np.unique(bloc)
blocs_test = hasard.choice(blocs_uniques, size=max(1, int(0.3 * len(blocs_uniques))), replace=False)
dans_test = np.isin(bloc, blocs_test)

tirage = []
for classe in (0, 1):
    candidats = np.flatnonzero(utilisable & (y_ref == classe))
    tirage.append(hasard.choice(candidats, size=min(PIXELS_PAR_CLASSE, len(candidats)), replace=False))
tirage = np.concatenate(tirage)

entrainement, test = tirage[~dans_test[tirage]], tirage[dans_test[tirage]]
X_train, y_train = X_ref[entrainement], y_ref[entrainement]
X_test, y_test = X_ref[test], y_ref[test]
print(f"Pixels d'entraînement : {len(y_train):,} (dont {int(y_train.sum()):,} bâtis)")
print(f"Pixels de test        : {len(y_test):,} (dont {int(y_test.sum()):,} bâtis)")

carte_tirage = np.zeros(H * W, dtype="uint8")
carte_tirage[entrainement], carte_tirage[test] = 1, 2
fig, ax = plt.subplots(figsize=(10, 10 * H / W))
ax.imshow(en_couleurs(images[ANNEE_REF]), alpha=0.5)
l, c = np.divmod(entrainement, W); ax.scatter(c, l, s=1, color="#1565c0", label="Entraînement")
l, c = np.divmod(test, W); ax.scatter(c, l, s=1, color="#ff6f00", label="Test")
ax.legend(markerscale=10, loc="lower right"); ax.set_axis_off()
ax.set_title("Où sont les pixels d'entraînement et de test ?")
plt.show()

## Étape 6 — Entraîner le modèle

> **Étape IA — Random Forest (forêt aléatoire).** Le modèle est un ensemble de 150 **arbres de décision**. Chaque arbre est une suite de questions simples (« le NDVI est-il inférieur à 0,2 ? », « le moyen infrarouge est-il supérieur à 0,25 ? ») qui aboutit à une réponse : bâti ou non bâti. Chaque arbre apprend sur un échantillon légèrement différent, puis **les 150 arbres votent**. Le pourcentage de votes « bâti » donne une **probabilité**.
>
> Personne n'écrit ces règles : l'algorithme les **déduit des exemples**. C'est la différence entre programmer et faire apprendre.

In [ ]:
modele = RandomForestClassifier(n_estimators=150, min_samples_leaf=3, n_jobs=-1, random_state=42)
debut = time.time()
modele.fit(X_train, y_train)
print(f"Modèle entraîné en {time.time() - debut:.1f} secondes sur {len(y_train):,} pixels.")
print(f"Nombre d'arbres : {len(modele.estimators_)}")
print(f"Nombre moyen de questions par arbre : {np.mean([a.tree_.node_count - a.tree_.n_leaves for a in modele.estimators_]):.0f}")

## Étape 7 — Évaluer le modèle

> **Étape IA — Matrice de confusion.** On demande au modèle de classer les pixels de test, puis on compare avec leurs étiquettes. La matrice de confusion compte les quatre cas possibles : bonnes réponses et erreurs dans chaque sens. On en tire trois chiffres :
> - **exactitude** : part des pixels bien classés ;
> - **rappel** : sur 100 pixels réellement bâtis, combien le modèle en retrouve ;
> - **précision** : sur 100 pixels que le modèle annonce bâtis, combien le sont vraiment.

In [ ]:
prediction_test = modele.predict(X_test)
mc = confusion_matrix(y_test, prediction_test, labels=[0, 1])
exactitude = np.trace(mc) / mc.sum()
rappel = mc[1, 1] / max(1, mc[1].sum())
precision = mc[1, 1] / max(1, mc[:, 1].sum())

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.imshow(mc, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{mc[i, j]:,}", ha="center", va="center", fontsize=16,
                color="white" if mc[i, j] > mc.max() / 2 else "black")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Non bâti", "Bâti"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["Non bâti", "Bâti"])
ax.set_xlabel("Réponse du modèle"); ax.set_ylabel("Étiquette de référence")
ax.set_title("Matrice de confusion (pixels de test)")
plt.tight_layout(); plt.show()

print(f"Exactitude : {exactitude:.1%} des pixels de test sont bien classés.")
print(f"Rappel     : sur 100 pixels réellement bâtis, le modèle en retrouve {100 * rappel:.0f}.")
print(f"Précision  : sur 100 pixels annoncés bâtis, {100 * precision:.0f} le sont vraiment.")
print(f"Erreurs    : {mc[0, 1]:,} fausses alertes (non bâti classé bâti), {mc[1, 0]:,} oublis (bâti non détecté).")

> **Étape IA — Ouvrir la boîte noire.** Le modèle indique quelles variables lui ont été les plus utiles pour décider. C'est un premier niveau d'**explicabilité** : on peut vérifier que le modèle s'appuie sur des critères qui ont un sens physique.

In [ ]:
importance = pd.Series(modele.feature_importances_, index=NOMS_VARIABLES).sort_values()
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(importance.index, 100 * importance.values, color="#1565c0")
ax.set_xlabel("Importance dans les décisions du modèle (%)")
ax.set_title("Quelles variables le modèle utilise-t-il le plus ?")
plt.tight_layout(); plt.show()

In [ ]:
# Contrôle visuel : la carte produite par le modèle sur l'année de référence
def probabilite_bati(image):
    X = variables(image)
    valide = np.isfinite(X).all(axis=1)
    p = np.full(len(X), np.nan, dtype="float32")
    if valide.any():
        p[valide] = modele.predict_proba(X[valide])[:, 1]
    return p.reshape(H, W)

proba_ref = probabilite_bati(images[ANNEE_REF])
fig, axes = plt.subplots(1, 3, figsize=(20, 6.8 * H / W + 0.6))
axes[0].imshow(en_couleurs(images[ANNEE_REF])); axes[0].set_title(f"Image Landsat {ANNEE_REF}")
axes[1].imshow(part_bati, cmap="Reds", vmin=0, vmax=1); axes[1].set_title("Référence : part de bâti")
vue = axes[2].imshow(proba_ref, cmap="Reds", vmin=0, vmax=1); axes[2].set_title("Notre modèle : probabilité de bâti")
for ax in axes:
    ax.set_axis_off()
fig.colorbar(vue, ax=axes, fraction=0.015, pad=0.01)
plt.show()

**Comparez les deux cartes de droite.** Où le modèle se trompe-t-il ? Regardez en particulier les plages, les sols nus, les carrières, les lits d'oued et les serres : pour un satellite, ils ressemblent beaucoup à du bâti.

## Étape 8 — Appliquer le modèle à 40 ans d'images

> **Étape IA — Prédiction (inférence).** Le modèle a appris sur l'image de l'année de référence. On lui présente maintenant, une par une, les images de toutes les autres années : pour chaque pixel, il donne la probabilité que ce soit du bâti. C'est ici que l'IA fait gagner du temps : des millions de pixels classés en moins de deux minutes.
>
> **Limite à garder en tête — la généralisation.** Le modèle a appris sur une image récente (satellite Landsat 8). Les images anciennes viennent d'autres capteurs (Landsat 5), prises dans d'autres conditions. Rien ne garantit qu'il soit aussi fiable en 1990 qu'à l'année de référence : nous n'avons pas d'étiquettes anciennes pour le vérifier.

In [ ]:
debut = time.time()
probas = np.stack([probabilite_bati(images[a]) for a in annees])      # (années, lignes, colonnes)
print(f"{probas.shape[0]} années x {H * W:,} pixels = {probas.shape[0] * H * W:,} pixels classés "
      f"en {time.time() - debut:.0f} secondes.")

surface_brute = np.array([np.nansum(p > 0.5) * HA_PAR_PIXEL for p in probas])
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(annees, surface_brute, "o-", color="grey")
ax.set_ylabel("Surface classée bâtie (ha)")
ax.set_title(f"{NOM_ZONE} : résultat brut du modèle, année par année")
ax.grid(alpha=0.3)
plt.show()

**Observez la courbe brute.** La tendance est là, mais la courbe est en dents de scie : la surface bâtie semble parfois diminuer d'une année à l'autre, ce qui n'a pas de sens. Le modèle classe chaque année **indépendamment**, sans savoir ce qu'il a répondu l'année précédente.

## Étape 9 — Ajouter une règle de bon sens : une zone bâtie le reste

> **Étape IA — Combiner le modèle et la connaissance métier.** Un modèle seul produit des résultats bruités. On ajoute une contrainte connue des urbanistes : un terrain passe de « non bâti » à « bâti » **une seule fois**. Pour chaque pixel, on cherche donc l'**année de bascule** qui s'accorde le mieux avec les 40 probabilités données par le modèle. Une erreur isolée (une année nuageuse, un sol nu très clair un été) ne suffit plus à changer le résultat.

In [ ]:
p = np.where(np.isfinite(probas), probas, 0.5).astype("float32")     # année sans donnée = pas d'avis
T = len(annees)
zero = np.zeros((1, H, W), dtype="float32")
cout_avant = np.concatenate([zero, np.cumsum(p, axis=0)])                    # erreurs si « non bâti » avant
cout_apres = np.concatenate([np.cumsum((1 - p)[::-1], axis=0)[::-1], zero])  # erreurs si « bâti » après
bascule = np.argmin(cout_avant + cout_apres, axis=0)       # 0 = déjà bâti au début, T = jamais bâti

surface_lissee = np.array([(bascule <= i).sum() * HA_PAR_PIXEL for i in range(T)])
gain = surface_lissee[-1] - surface_lissee[0]
duree = annees[-1] - annees[0]

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(annees, surface_brute, "o", color="lightgrey", label="Modèle brut, année par année")
ax.plot(annees, surface_lissee, "-", color="#c4281b", linewidth=3, label="Avec la règle « une zone bâtie le reste »")
ax.set_ylabel("Surface bâtie (ha)")
ax.set_title(f"{NOM_ZONE} : évolution de la surface bâtie, {annees[0]}-{annees[-1]}")
ax.grid(alpha=0.3); ax.legend()
plt.show()

print(f"===== {NOM_ZONE} : bilan {annees[0]} - {annees[-1]} =====")
print(f"Surface bâtie en {annees[0]} : {surface_lissee[0]:,.0f} ha")
print(f"Surface bâtie en {annees[-1]} : {surface_lissee[-1]:,.0f} ha")
if surface_lissee[0] > 0:
    print(f"La surface bâtie a été multipliée par {surface_lissee[-1] / surface_lissee[0]:.1f}")
print(f"Rythme moyen : {gain / duree:,.0f} ha par an, soit {gain / duree / 0.714 / 365:.1f} terrain(s) de football par jour")

### La carte de l'âge des quartiers
Chaque pixel bâti est coloré selon la période où le modèle l'a vu basculer en « bâti ».

In [ ]:
annee_bati = np.zeros((H, W), dtype="uint16")              # 0 = non bâti
construit = bascule < T
annee_bati[construit] = np.array(annees)[bascule[construit]]

PERIODES = [(f"Déjà bâti en {annees[0]}", annees[0], annees[0], (120, 120, 120)),
            (f"{annees[0] + 1}-1994", annees[0] + 1, 1994, (255, 221, 0)),
            ("1995-2004", 1995, 2004, (255, 145, 0)),
            ("2005-2014", 2005, 2014, (229, 28, 35)),
            (f"2015-{annees[-1]}", 2015, annees[-1], (142, 36, 170))]
PERIODES = [per for per in PERIODES if per[1] <= per[2]]

fond = en_couleurs(images[annees[-1]]).mean(axis=-1, keepdims=True).repeat(3, axis=-1).astype("uint8")
carte_age = fond.copy()
surfaces_periodes = {}
for nom, debut_p, fin_p, couleur in PERIODES:
    masque = (annee_bati >= debut_p) & (annee_bati <= fin_p)
    carte_age[masque] = couleur
    surfaces_periodes[nom] = masque.sum() * HA_PAR_PIXEL

fig, ax = plt.subplots(figsize=(14, 14 * H / W))
ax.imshow(carte_age, interpolation="nearest")
ax.legend(handles=[Patch(facecolor=np.array(c) / 255, label=f"{n} : {surfaces_periodes[n]:,.0f} ha")
                   for n, _, _, c in PERIODES], loc="lower right", fontsize=11)
ax.set_title(f"{NOM_ZONE} : période de construction détectée par le modèle", fontsize=14)
ax.set_axis_off()
plt.savefig("carte_age_du_bati.png", dpi=150, bbox_inches="tight")
plt.show()

### L'animation de l'étalement urbain
La même animation qu'au début, mais cette fois **le modèle montre en rouge ce qu'il a détecté comme nouveau bâti**, avec le compteur d'hectares.

In [ ]:
vues, textes = [], []
for i, a in enumerate(annees):
    vue = en_couleurs(images[a]).astype("float32")
    nouveau = (bascule > 0) & (bascule <= i)
    vue[nouveau] = 0.35 * vue[nouveau] + 0.65 * np.array([255, 0, 0])
    vues.append(vue.astype("uint8"))
    textes.append(f"{a}   +{surface_lissee[i] - surface_lissee[0]:,.0f} ha")
gif_ia = creer_gif(vues, textes, "etalement_urbain_IA.gif", titre=f"{NOM_ZONE} - extension urbaine vue par l'IA", ips=3)
display(Image(filename=gif_ia))

### Carte interactive
Zoomez sur votre quartier. Avec l'icône en haut à droite, cochez ou décochez les périodes.
*(Le fond satellite est récent : il sert à vérifier ce qui existe aujourd'hui.)*

In [ ]:
def vers_carte_web(calque_rvba, grille):
    transfo, largeur, hauteur = calculate_default_transform(
        grille["crs"], "EPSG:3857", grille["width"], grille["height"], *grille["bounds"])
    sortie = np.zeros((4, hauteur, largeur), dtype="uint8")
    reproject(np.moveaxis(calque_rvba, -1, 0), sortie, src_transform=grille["transform"],
              src_crs=grille["crs"], dst_transform=transfo, dst_crs="EPSG:3857",
              resampling=Resampling.nearest)
    o, s, e, n = transform_bounds("EPSG:3857", "EPSG:4326", *array_bounds(hauteur, largeur, transfo))
    return np.moveaxis(sortie, 0, -1), [[s, o], [n, e]]

carte = folium.Map(location=[(bbox[1] + bbox[3]) / 2, (bbox[0] + bbox[2]) / 2], zoom_start=12,
                   tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
                   attr="Esri World Imagery")
for nom, debut_p, fin_p, couleur in PERIODES:
    rvba = np.zeros((H, W, 4), dtype="uint8")
    rvba[(annee_bati >= debut_p) & (annee_bati <= fin_p)] = couleur + (255,)
    image, limites = vers_carte_web(rvba, GRILLE_30)
    folium.raster_layers.ImageOverlay(image=image, bounds=limites, opacity=0.75, name=nom).add_to(carte)
folium.LayerControl(collapsed=False).add_to(carte)
carte

**Vérifiez le résultat sur un quartier que vous connaissez :**
- La période annoncée par le modèle correspond-elle à ce que vous savez ?
- Trouvez une erreur : une zone colorée qui n'est pas du bâti (plage, carrière, sol nu, lit d'oued), ou un quartier oublié.

## Étape 10 — Exporter vers un SIG
Les fichiers apparaissent dans le panneau **Fichiers** à gauche : clic droit → Télécharger.
- `annee_construction_….tif` : pour chaque pixel, l'année de bascule en bâti (0 = non bâti), à ouvrir dans QGIS ou ArcGIS ;
- `surface_batie_….csv` : la courbe année par année.

In [ ]:
nom = NOM_ZONE.lower().replace(" ", "_")
fichier_tif = f"annee_construction_{nom}.tif"
with rasterio.open(fichier_tif, "w", driver="GTiff", height=H, width=W, count=1, dtype="uint16",
                   crs=GRILLE_30["crs"], transform=GRILLE_30["transform"], nodata=0,
                   compress="deflate") as sortie:
    sortie.write(annee_bati, 1)

courbe = pd.DataFrame({"Année": annees, "Surface bâtie brute (ha)": surface_brute.round(1),
                       "Surface bâtie lissée (ha)": surface_lissee.round(1)})
courbe.to_csv(f"surface_batie_{nom}.csv", index=False)
print("Fichiers créés :")
for f in (fichier_tif, f"surface_batie_{nom}.csv", "carte_age_du_bati.png",
          "etalement_urbain_IA.gif", "timelapse_landsat.gif"):
    print(" -", f)

## Récapitulatif : la chaîne d'IA que vous venez de dérouler

| Étape | Terme technique | Ce que vous avez fait |
|---|---|---|
| 2 | Données | 40 images annuelles Landsat à 6 bandes |
| 3 | Étiquettes | Bâti / non bâti, tirés d'une carte produite par Deep Learning |
| 4 | Variables | 6 bandes + 3 indices pour décrire chaque pixel |
| 5 | Entraînement / test | Blocs de 1 km réservés à l'examen du modèle |
| 6 | Apprentissage | Random Forest de 150 arbres |
| 7 | Évaluation | Matrice de confusion, exactitude, rappel, précision, importance des variables |
| 8 | Inférence | Le modèle classe toutes les années depuis 1984 |
| 9 | Post-traitement | Règle métier « une zone bâtie le reste » |

## Questions de synthèse (à discuter en groupe)
1. Le modèle a appris à partir d'une carte elle-même produite par une IA, exacte à un peu plus de 75 %. Que mesure alors vraiment notre matrice de confusion ? Comment obtenir une meilleure « vérité » ?
2. Le modèle a été entraîné sur une image récente et appliqué à 1984. Quels indices, dans vos résultats, montrent qu'il généralise bien ou mal ? Comment le vérifier (photos aériennes anciennes, plans, cadastre) ?
3. Quelles confusions avez-vous observées (sol nu, plage, carrière, serres) ? Quelle variable supplémentaire aiderait le modèle (relief, texture, lumières nocturnes, radar) ?
4. Changez `PIXELS_PAR_CLASSE` à l'étape 5 (500, puis 20 000) et relancez les étapes 5 à 7. Que devient l'exactitude ? Qu'en concluez-vous sur la quantité de données nécessaire ?
5. Quelle est la différence entre ce modèle (Random Forest, pixel par pixel) et un modèle de Deep Learning comme celui du TP 1 ou celui des empreintes de bâtiments vu lundi ?
6. Comment votre administration pourrait-elle utiliser la courbe et la carte de l'âge du bâti ? Quelles vérifications faudrait-il avant de publier un chiffre officiel ?

---
**À retenir**
- Un modèle d'IA supervisé, c'est : des exemples étiquetés, des variables, un apprentissage, puis une évaluation sur des données jamais vues.
- Le modèle ne vaut pas mieux que ses étiquettes, et rien ne garantit qu'il reste fiable hors des conditions où il a appris.
- Les meilleurs résultats viennent de la combinaison du modèle et de la connaissance métier.

*Données : Landsat Collection 2 (USGS/NASA) ; 10m Annual Land Use Land Cover (Impact Observatory, Esri, Microsoft ; licence CC-BY 4.0), via Microsoft Planetary Computer.*